In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import select, insert
from sqlalchemy.orm import Session
import polars as pl

# Importing from 'app' module
from app.config import db_engine
from app.models import SilverCleanAd, GoldMarketBaseline

In [ ]:
with db_engine.connect() as connection:
    df_silver_raw = pl.read_database(
        select(
            SilverCleanAd.category,
            SilverCleanAd.cpu_brand,
            SilverCleanAd.ram_gb,
            SilverCleanAd.storage_gb,
            SilverCleanAd.ad_id,
            SilverCleanAd.price,
            SilverCleanAd.date,
            SilverCleanAd.baseline_id
        ),
        connection=connection
    )

In [ ]:
market_baselines_df = (
    df_silver_raw
    # Only the max date over baseline_id
    .filter(
        pl.col('date') == pl.col('date').max().over('baseline_id')
    )
    .group_by(['baseline_id', 'category', 'cpu_brand', 'ram_gb', 'storage_gb'])
    .agg(
        pl.col('ad_id').count().alias('active_ads_count'),
        pl.col('price').min().alias('min_price'),
        pl.col('price').median().alias('median_price'),
        pl.col('price').max().alias('max_price'),
        pl.col('date').max().alias('last_recalculated_at'),
    )
)

In [ ]:
if not market_baselines_df.is_empty():
    with Session(db_engine) as session:
        session.execute(
            insert(GoldMarketBaseline), market_baselines_df.to_dicts()
        )
else:
    print("No data found to insert.")